# Calibre → Reading List sync

Pull books from a [Calibre](https://calibre-ebook.com/) library and rebuild `data/books.json`, which feeds the Reading List page (`web/books.html`).

The two halves are deliberately separate so the slow part can run **asynchronously**, decoupled from building the project:

1. **Pull** — `build_books.fetch_calibre_books_async(...)` awaits the `calibredb` subprocess. Slow (talks to Calibre), so we run it as a background task.
2. **Build** — `build_books.build_books_json(records=...)` maps the pulled records, merges them onto the curated overlay, and writes `data/books.json`. Fast and pure.
3. **Ship** — `build_site.main()` assembles the static `site/`.

**Requirements:** Calibre installed with `calibredb` on `PATH`. Run this notebook from the repo (`jupyter lab` in the repo root).

## Setup

Make the repo importable and point at your library. Leave `LIBRARY = None` to use Calibre's default local library; set a folder path, or a Content Server URL like `"http://host:8080/#Library"`, to target another.

In [ ]:
import os, sys, asyncio

# repo root = parent of notebooks/ ; make build_books / build_site importable
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, REPO)

import build_books
import build_site

LIBRARY = None       # None = default local library; or "/path/to/library"; or "http://host:8080/#Library"
CALIBREDB = "calibredb"
print("repo:", REPO)

## Step 1 — kick off the Calibre pull asynchronously

`fetch_calibre_books_async` awaits the `calibredb` subprocess, so it yields control instead of blocking. We schedule it as a task and let it run in the background while we do other work.

In [ ]:
# schedule the pull; returns immediately with a running Task
pull = asyncio.ensure_future(
    build_books.fetch_calibre_books_async(library=LIBRARY, calibredb=CALIBREDB)
)
print("pull started:", pull)

In [ ]:
# ... the pull runs in the background. Do other work here if you like ...
# When you want the result, await it (Jupyter supports top-level await):
records = await pull
print(f"pulled {len(records)} book(s) from Calibre")
records[:1]  # peek at one raw --for-machine record

## Step 2 — build `books.json` from the pulled records

Passing `records=` skips a second Calibre call — we reuse what Step 1 already pulled. This maps Calibre's facts (title, author, year, rating, blurb, tags→topics, added) and **preserves curated reading state** (status / pages_read / started / finished) from the existing `data/books.json`, unless Calibre custom columns supply it.

Use `dry_run=True` first to preview without writing.

In [ ]:
# preview
preview = build_books.build_books_json(records=records, dry_run=True)
for b in preview[:10]:
    print(f"  {b['status']:11s} {b['rating'] or '-'}  {b['title']}  — {b['author']}")

In [ ]:
# write data/books.json for real
books = build_books.build_books_json(records=records)
print(f"wrote data/books.json ({len(books)} books)")

## Step 3 — build the site (decoupled from the pull)

The site build is independent of the Calibre pull — it just consumes the `books.json` we wrote. Run it in a thread executor so a long build doesn't block the notebook's event loop.

In [ ]:
loop = asyncio.get_event_loop()
await loop.run_in_executor(None, build_site.main)
print("site/ rebuilt — preview:  cd ../site && python -m http.server 8000")

## One-shot: pull + build concurrently

The whole flow as a single awaitable — the Calibre pull runs concurrently with any other async work you schedule, then the (synchronous, pure) build + site steps run once the records are in.

In [ ]:
async def sync_and_build(library=LIBRARY):
    records = await build_books.fetch_calibre_books_async(library=library, calibredb=CALIBREDB)
    books = build_books.build_books_json(records=records)
    await asyncio.get_event_loop().run_in_executor(None, build_site.main)
    return books

books = await sync_and_build()
print(f"done — {len(books)} books synced and site rebuilt")